# 13 — MCP Client (`core/mcp_client.py`)
MCP (Model Context Protocol) client factory for loading Collibra and Jira tools as LangChain `BaseTool` objects.

- **`get_mcp_tools(server_name)`** → returns tool list, or `[]` when disabled/unavailable
- **`is_mcp_enabled()`** → bool  
- **`list_configured_servers()`** → server names with env vars set

Agents call `get_mcp_tools("collibra")` at init — if empty, fall back to REST mock.  
**Default: `USE_MCP=false`** — always returns `[]`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)
os.environ["USE_MCP"] = "false"  # default

## 1. Default State — MCP Disabled

In [ ]:
from core.mcp_client import get_mcp_tools, is_mcp_enabled, list_configured_servers

print("MCP enabled:", is_mcp_enabled())
print("Collibra tools:", get_mcp_tools("collibra"))  # []
print("Jira tools    :", get_mcp_tools("jira"))       # []
print("Configured    :", list_configured_servers())   # []

## 2. Enable MCP (without actual binary configured)

In [ ]:
os.environ["USE_MCP"] = "true"

# Reload the module to pick up the env var change
import importlib, core.mcp_client as mcp_mod
mcp_mod._USE_MCP = True  # patch directly

print("MCP enabled:", mcp_mod.is_mcp_enabled())

# No server path configured → returns []
tools = mcp_mod.get_mcp_tools("collibra")
print("Collibra tools (no binary configured):", tools)

os.environ["USE_MCP"] = "false"
mcp_mod._USE_MCP = False

## 3. Server Registry

In [ ]:
# _SERVER_ENV maps server name → environment variable
from core.mcp_client import _SERVER_ENV
print("MCP server registry:")
for server, env_var in _SERVER_ENV.items():
    configured = bool(os.getenv(env_var, ""))
    print(f"  {server:<15} → env var: {env_var:<30} | configured: {configured}")

## 4. How Agents Use MCP (Pattern)

In [ ]:
# This is the pattern used in MetadataAgent and CapacityAgent __init__

def init_agent_with_mcp_fallback(server_name: str):
    from core.mcp_client import get_mcp_tools
    mcp_tools = get_mcp_tools(server_name)
    
    if mcp_tools:
        mode = "MCP"
        client = None
        print(f"  → Using MCP tools: {[t.name for t in mcp_tools]}")
    else:
        mode = "REST fallback"
        client = f"Mock{server_name.title()}Client()"  # would be real client in prod
        print(f"  → Using REST client: {client}")
    
    return mode, client

print("MetadataAgent init:")
mode, client = init_agent_with_mcp_fallback("collibra")
print(f"  Mode: {mode}, Client: {client}")

print("\nCapacityAgent init:")
mode, client = init_agent_with_mcp_fallback("jira")
print(f"  Mode: {mode}, Client: {client}")

## 5. list_configured_servers — Only When Env Vars Set

In [ ]:
# Without any MCP server paths configured
print("Configured servers (before):", list_configured_servers())

# Simulate having a server configured
os.environ["COLLIBRA_MCP_SERVER"] = "/path/to/collibra-mcp"
mcp_mod._USE_MCP = True
print("Configured servers (after):", mcp_mod.list_configured_servers())

# Cleanup
del os.environ["COLLIBRA_MCP_SERVER"]
mcp_mod._USE_MCP = False

## 6. Production Setup (Reference)

In [ ]:
print("Production MCP .env config:"
      + chr(10) + "USE_MCP=true"
      + chr(10) + "COLLIBRA_MCP_SERVER=/opt/mcp-servers/collibra-mcp-server"
      + chr(10) + "JIRA_MCP_SERVER=/opt/mcp-servers/jira-mcp-server")